# 10 • Tool Calling: ราคาหุ้นศรีตรังและสถานะคลัง

เป้าหมาย: ดูชื่อ tool และ arguments ที่ LLM เลือก รัน Python tool แล้วส่งผลกลับไปให้ LLM ตอบภาษาไทย

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ กฏ ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/biodatlab/sritrang-ai-workshop/blob/main/10_tool_calling.ipynb)

เครื่องมือราคาหุ้นอ่านข้อมูลจาก Yahoo Finance โดยไม่ต้องใช้ API key ของ Yahoo ข้อมูลอาจล่าช้า และ endpoint อาจจำกัดคำขอหรือเปลี่ยนแปลงได้


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"

INSTRUCTIONS = "ตอบคำถามเป็นภาษาไทย ใช้เครื่องมือเมื่อจำเป็น อ้างอิงข้อมูลที่ได้รับ หากข้อมูลไม่เพียงพอหรืออ่านข้อมูลไม่ได้ให้บอกตามจริง"


In [ ]:
def get_flood_level(factory):
    readings = {"คลังยางแท่ง A": 55, "คลังน้ำยาง B": 10}
    if factory not in readings:
        return {"error": "ไม่พบหน่วยงานนี้", "source": "ข้อมูลสมมติสำหรับการอบรม"}
    return {"factory": factory, "water_cm": readings[factory], "source": "ข้อมูลสมมติสำหรับการอบรม", "live": False}

from datetime import datetime, timezone, timedelta
from urllib.request import Request, urlopen
import json

def get_stock_price(symbol: str = "STA.BK"):
    """อ่านราคาหุ้นล่าสุดที่ Yahoo Finance รายงานสำหรับ Sri Trang Agro-Industry (STA.BK) พร้อมสกุลเงินและเวลาของราคา ข้อมูลอาจล่าช้า"""
    if symbol != "STA.BK":
        return {"error": "เครื่องมือนี้รองรับ STA.BK เท่านั้น", "symbol": symbol}
    url = "https://query1.finance.yahoo.com/v8/finance/chart/STA.BK?interval=1d&range=1d"
    try:
        request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request, timeout=20) as response:
            data = json.load(response)
        chart = data["chart"]
        if chart.get("error") or not chart.get("result"):
            return {"error": "Yahoo Finance ไม่ส่งข้อมูลราคา", "symbol": symbol}
        meta = chart["result"][0]["meta"]
        price = meta.get("regularMarketPrice")
        timestamp = meta.get("regularMarketTime")
        if price is None or timestamp is None:
            return {"error": "ข้อมูลราคาหรือเวลาไม่ครบ", "symbol": symbol}
        quote_time = datetime.fromtimestamp(timestamp, timezone(timedelta(hours=7)))
        return {
            "symbol": symbol,
            "company": meta.get("longName", "Sri Trang Agro-Industry"),
            "price": price,
            "currency": meta.get("currency"),
            "quote_time": quote_time.isoformat(),
            "source": "Yahoo Finance",
            "source_url": "https://finance.yahoo.com/quote/STA.BK/",
            "note": "ราคาล่าสุดที่แหล่งข้อมูลรายงาน อาจล่าช้า ไม่ใช่ข้อมูลสมมติ",
        }
    except (OSError, ValueError, KeyError, IndexError, TypeError):
        return {"error": "อ่านข้อมูลจาก Yahoo Finance ไม่สำเร็จ กรุณาลองใหม่", "symbol": symbol}


In [ ]:
tools = [{
    "type": "function", "name": "get_flood_level",
    "description": "อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B",
    "parameters": {"type": "object", "properties": {
        "factory": {"type": "string", "enum": ["คลังยางแท่ง A", "คลังน้ำยาง B"]}
    }, "required": ["factory"], "additionalProperties": False},
    "strict": True,
}]
tools.append({
    "type": "function", "name": "get_stock_price",
    "description": "อ่านราคาหุ้น Sri Trang Agro-Industry (STA.BK) จาก Yahoo Finance พร้อมสกุลเงินและเวลาของราคา ข้อมูลอาจล่าช้า",
    "parameters": {"type": "object", "properties": {
        "symbol": {"type": "string", "enum": ["STA.BK"]}
    }, "required": ["symbol"], "additionalProperties": False},
    "strict": True,
})
functions = {"get_flood_level": get_flood_level, "get_stock_price": get_stock_price}


## 1. ระดับน้ำของบริษัท


In [ ]:
messages = [{"role": "user", "content": "ระดับน้ำสมมติที่คลังยางแท่ง A ของบริษัทศรีตรังเป็นเท่าไร?"}]
response = client.responses.create(
    model=MODEL, input=messages, tools=tools, instructions=INSTRUCTIONS,
    tool_choice="auto", parallel_tool_calls=False,
    max_output_tokens=500,
)
messages += response.output
for call in response.output:
    if call.type == "function_call":
        print("Tool ที่เลือก:", call.name, "Arguments:", call.arguments)
        result = functions[call.name](**json.loads(call.arguments))
        print("ผลลัพธ์:", result)
        messages.append({"type": "function_call_output", "call_id": call.call_id,
                         "output": json.dumps(result, ensure_ascii=False)})


In [ ]:
final = client.responses.create(
    model=MODEL, input=messages,
    instructions=INSTRUCTIONS,
    max_output_tokens=500,
)
print(final.output_text)


### ตัวอย่างคำตอบ
> ระดับน้ำที่คลังยางแท่ง A คือ 55 ซม. เป็นข้อมูลสมมติสำหรับการอบรม


## 2. ราคาหุ้นของบริษัท


In [ ]:
messages = [{"role": "user", "content": "ราคาหุ้นของบริษัทศรีตรัง STA ล่าสุดที่ Yahoo Finance รายงานเท่าไร? ระบุสกุลเงินและเวลาของราคา"}]
response = client.responses.create(
    model=MODEL, input=messages, tools=tools, instructions=INSTRUCTIONS,
    tool_choice="auto", parallel_tool_calls=False,
    max_output_tokens=500,
)
messages += response.output
for call in response.output:
    if call.type == "function_call":
        print("Tool ที่เลือก:", call.name, "Arguments:", call.arguments)
        result = functions[call.name](**json.loads(call.arguments))
        print("ผลลัพธ์:", result)
        messages.append({"type": "function_call_output", "call_id": call.call_id,
                         "output": json.dumps(result, ensure_ascii=False)})


In [ ]:
final = client.responses.create(
    model=MODEL, input=messages,
    instructions=INSTRUCTIONS,
    max_output_tokens=500,
)
print(final.output_text)


### ตัวอย่างคำตอบ
> ราคาหุ้น STA.BK คือ {price} {currency} ณ {quote_time} จาก Yahoo Finance ข้อมูลอาจล่าช้า


## ลองทำด้วยตนเอง
1. ถามราคาหุ้นศรีตรัง แล้วตรวจว่าโมเดลเลือก `get_stock_price` พร้อม symbol `STA.BK`
2. ถามระดับน้ำสมมติที่คลังน้ำยาง B แล้วดูว่าโมเดลเปลี่ยนไปใช้ `get_flood_level`
3. ลองทักทาย แล้วสังเกตว่าโมเดลจำเป็นต้องเรียก tool หรือไม่
4. ตรวจชื่อ tool, arguments และผลลัพธ์ก่อนอ่านคำตอบสุดท้าย

ราคาหุ้นมาจาก Yahoo Finance และอาจล่าช้า ส่วนระดับน้ำเป็นข้อมูลสมมติสำหรับการอบรม
